# Image Compression, Reconstruction, and Facial Feature Analysis via PCA & SVD

**Objective:** Using the famous Lenna image for experiments, the project performs data compression, cutting and reassembling with different patch sizes, digital image compression and generation, as well as examining the encryption and decryption effects on facial and non-facial images. It also explores the feasibility of using facial features to encrypt non-facial images, while observing the quality of these compressed results and analyzing the underlying factors.

<hr>

## Part 1:
Compress an image $X$ using SVD's "Rank $q$ approximation" while maintaining image quality. Compare the following reorganization arrangements for the image matrix $X$ before performing the "Rank $q$ approximation." Under the same compression ratio, observe which arrangement yields the best restored image quality, and explain why.

### 1.1 $X$ unchanged

In [ ]:
import numpy as np
from numpy.linalg import svd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

imgfile = "lenna.png" # 512x512x3
X = mpimg.imread(imgfile)
if len(X.shape) > 2:
    X = np.mean(X, axis=2) # convert RGB to grayscale
N, p = X.shape
U, E, VT = svd(X, full_matrices=False)
q = np.array([p/4, p/8, p/16]).astype('int') # q = (128,64,32)
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for i, r in enumerate(q):
    Xq = U[:, :r] @ np.diag(E[:r]) @ VT[:r, :]
    ax[i].imshow(Xq, cmap = 'gray') # plot a figure
    ax[i].set_title('Compression ratio: {}'.format(p/r/2))
    ax[i].set_xticks([])
    ax[i].set_yticks([])
plt.show()

### 2.2 Segment the image $X$ into $8\times 8$ patches, flatten each patch into a $64\times 1$ vector, and then rearrange these vectors to form a new $64\times N$ matrix.

In [ ]:
# segment an image
def reshape_graph(X, patch_size):
    N, p = X.shape
    p_patch = patch_size**2
    N_patch = int(N*p/ p_patch)
    M = np.zeros((N_patch, patch_size**2))
    for i in range(int(N/ patch_size)):
        for j in range(int(p/ patch_size)):
            M[i*p//patch_size+j, :] = X[i*patch_size:(i+1)*patch_size, j*patch_size:(j+1)*patch_size].reshape(1,-1)
    return M.T

In [ ]:
def montage(A, m, n):
    '''
    Create a montage matrix with mn images
    Inputs:
    A: original pxN image matrix with N images (p pixels), N > mn
    m, n: m rows & n columns, total mn images
    Output:
    M: montage matrix containing mn images
    '''
    sz = np.sqrt(A.shape[0]).astype('int') # image size sz x sz
    M = np.zeros((m*sz, n*sz)) # montage image
    for i in range(m) :
        for j in range(n) :
            M[i*sz: (i+1)*sz, j*sz:(j+1)*sz] = \
            A[:, i*n+j].reshape(sz, sz)
    return M

In [ ]:
import numpy as np
from numpy.linalg import svd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

def reshape_and_reform(patch_size):
    imgfile = "lenna.png" # 512x512x3
    X = mpimg.imread(imgfile)
    if len(X.shape) > 2:
        X = np.mean(X, axis=2) # convert RGB to grayscale

    N, p = X.shape
    X_ = reshape_graph(X,patch_size)
    U, E, VT = svd(X_, full_matrices=False)
    q = np.array([p/4, p/8, p/16]).astype('int')
    fig, ax = plt.subplots(1, 3, figsize=(16, 8))

    for i, r in enumerate(q):
        Xq = U[:, :r] @ np.diag(E[:r]) @ VT[:r, :]
        Xq_ = montage(Xq,N//patch_size,p//patch_size)
        ax[i].imshow(Xq_, cmap = 'gray')
        ax[i].set_title('Compression ratio: {}'.format(p/r/2))
        ax[i].set_xticks([])
        ax[i].set_yticks([])
    plt.show()

In [ ]:
reshape_and_reform(8)

### 1.3 Similarly, with a patch size of $16\times 16$ ($16\times 16$ per patch).

In [ ]:
reshape_and_reform(16)

### 1.4 Similarly, with a patch size of $32\times 32$ ($32\times 32$ per patch). 

In [ ]:
reshape_and_reform(32)

### Present them together, showing the same compression ratio per row while displaying different per patches across the columns.

In [ ]:
import numpy as np
from numpy.linalg import svd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

imgfile = "lenna.png" # 512x512x3
X = mpimg.imread(imgfile)
if len(X.shape) > 2:
    X = np.mean(X, axis=2) # convert RGB to grayscale

N, p = X.shape
q = np.array([p/4, p/8, p/16]).astype('int')
s = np.array([8,16,32])

for i in range((len(q))):
    fig, ax = plt.subplots(1, 4, figsize=(16, 8))
    U, E, VT = svd(X, full_matrices=False)
    Xq = U[:, :q[i]] @ np.diag(E[:q[i]]) @ VT[:q[i], :]
    ax[0].imshow(Xq, cmap = 'gray') 
    ax[0].set_title('original')
    ax[0].set_xticks([])
    ax[0].set_yticks([])
    for j,r in enumerate(s):
        X1_ = reshape_graph(X,r)
        U1, E1, VT1 = svd(X1_, full_matrices=False)
        Xq1 = U1[:, :q[i]] @ np.diag(E1[:q[i]]) @ VT1[:q[i], :]
        Xq1_ = montage(Xq1,int(N//r),int(p//r))
        ax[j+1].imshow(Xq1_, cmap = 'gray') 
        ax[j+1].set_title(f'{r}x{r} per patch')
        ax[j+1].set_xticks([])
        ax[j+1].set_yticks([])
    ax[0].set_ylabel(f'Compression ratio: {p/q[i]/2}', fontsize=12, fontweight='bold', rotation=0, labelpad=80, verticalalignment='center')
plt.show()

#### Conclusion:

* Comparison between Full-Image SVD and Patch-Based SVD:
Direct global SVD compression of the original image tends to cause global detail blurring. In contrast, the patch-based strategy effectively leverages the "local low-rank property" of images, achieving superior detail retention under the same compression ratio.

* Impact of Patch Size on Visual Performance:
  * $8 \times 8$ per patch: With a smaller cropping scope, they can precisely capture high-frequency edges and micro-textures while preventing multiple complex textures from mixing into the same block, resulting in cleaner and sharper reconstructed edges.
  * $16 \times 16$ per patch: Their visual performance is comparable to $8 \times 16$ ($8 \times 8$), but due to a larger dimension, they can capture a broader range of spatial continuity and gradient structures, performing better in overall structural fluency.
  * $32 \times 32$ per patch: Because the block size is too large, they frequently encompass multiple properties simultaneously (such as hair, hat edges, and background overlapping). This causes the blocks to lose their simple low-rank characteristics, leading to high-frequency details being averaged out during SVD approximation, which consequently induces noticeable blurring and degradation.

<hr>

## Part 2:
Taking 70,000 handwritten images as an example—with approximately 7,000 images per digit—write a piece of code to observe the images and quality of these handwritten digits, and ensure that a different image can be randomly viewed upon each execution.

In [ ]:
def montage(A, m, n):
    '''
    Create a montage matrix with mn images
    Inputs:
    A: original pxN image matrix with N images (p pixels), N > mn
    m, n: m rows & n columns, total mn images
    Output:
    M: montage matrix containing mn images
    '''
    sz = np.sqrt(A.shape[0]).astype('int') # image size sz x sz
    M = np.zeros((m*sz, n*sz)) # montage image
    for i in range(m) :
        for j in range(n) :
            M[i*sz: (i+1)*sz, j*sz:(j+1)*sz] = \
            A[:, i*n+j].reshape(sz, sz)
    return M

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

mnist = loadmat("mnist-original.mat")
X = mnist["data"]
y = mnist["label"][0]

fig, ax = plt.subplots(5, 2, figsize=(10, 15))
ax = ax.flatten()

for i in range(5*2):
    indices = np.where(y == i)[0]
    selected_idx = np.random.choice(indices, size=50, replace=False)
    selected_images = X[:,selected_idx]
    montage_image = montage(selected_images, 5, 10)
    ax[i].imshow(montage_image, cmap = 'gray', interpolation = 'nearest')
    # axs[i].axis('off')  
    ax[i].set_xticks([])
    ax[i].set_yticks([])
plt.show()

### Conclusion:
Before processing large-scale images, it is necessary to inspect the image plots to ensure a thorough understanding of the images to be processed and their data modalities. The aforementioned code enables the random observation of different handwritten digit images upon each execution.

<hr>

## Part 3:
In programming, "Rank-$q$ approximation" can be implemented using existing PCA or SVD libraries. Taking digit images as an example, this problem selects all images of a specific digit to perform a "Rank-$q$ approximation" in order to obtain a set of approximate images with the same size and number of images. To observe the effect of the $q$ value on the images, set $q$ to $1, 3, 5, \dots, 29$, and print out 20 images after "Rank-$q$ approximation" for each $q$ value, as shown in the figure below. The first row shows the original images, the second row corresponds to $q=1$, and the last row corresponds to $q=29$. Please use PCA and SVD libraries respectively to produce the same results. How should the program be written for each?

<p align="left">
  <img src="The montage of handwriting digits.png" width="600">
</p>

### 3.1 Using SVD:

In [ ]:
def montage(A, m, n):
    '''
    Create a montage matrix with mn images
    Inputs:
    A: original pxN image matrix with N images (p pixels), N > mn
    m, n: m rows & n columns, total mn images
    Output:
    M: montage matrix containing mn images
    '''
    sz = np.sqrt(A.shape[0]).astype('int') # image size sz x sz
    M = np.zeros((m*sz, n*sz)) # montage image
    for i in range(m) :
        for j in range(n) :
            M[i*sz: (i+1)*sz, j*sz:(j+1)*sz] = \
            A[:, i*n+j].reshape(sz, sz)
    return M

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from sklearn.decomposition import PCA

mnist = loadmat('mnist-original.mat')
X = mnist['data']   # shape = (784, 70000)
y = mnist['label'][0]

target_digit = 3
X_digit = X[:, y == target_digit]

np.random.seed(42)
selected_indices = np.random.choice(X_digit.shape[1], 40, replace=False)
X_ = X_digit[:, selected_indices]  # shape = (784, 40)

# set the range of q values：1, 3, 5, ..., 29
q_values = list(range(1, 30, 2))

# Centering SVD
mean_face = X_.mean(axis=1, keepdims=True)
X_centered = X_ - mean_face
U, E, VT = np.linalg.svd(X_centered, full_matrices=False)

montage_rows = []
montage_rows.append(montage(X_, 1, 20))

for q in q_values:
    U_q = U[:, :q]
    E_q = np.diag(E[:q])
    VT_q = VT[:q, :]
    X_approx = (U_q @ E_q @ VT_q) + mean_face
    
    montage_rows.append(montage(X_approx, 1, 20))

final_image = np.vstack(montage_rows)

plt.figure(figsize=(10, 12))
plt.imshow(final_image, cmap='gray')
plt.title("The Montage of handwriting digits", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

### 3.2 Using PCA:

In [ ]:
from sklearn.decomposition import PCA

# Since scikit-learn's PCA expects samples as rows and features as columns, we transpose the data to (20, 784).
q_values = list(range(1, 30, 2))

montage_rows = []
montage_rows.append(montage(X_, 1, 20))

X_T = X_.T

for i, q in enumerate(q_values):
    pca = PCA(n_components=q)
    X_pca = pca.fit_transform(X_T)
    X_pca_T = pca.inverse_transform(X_pca) # reshape back to the original shape (784, 20)
    
    # Note: sklearn's inverse_transform automatically adds back pca.mean_
    X_pca_ = X_pca_T.T
    montage_rows.append(montage(X_pca_, 1, 20))

final_image = np.vstack(montage_rows)

plt.figure(figsize = (10,12))
plt.imshow(final_image, cmap = 'gray')
plt.title("The Montage of handwriting digits", fontsize=14)
plt.axis('off')
plt.tight_layout()
plt.show()

#### Conclusion:

* **Separating Visualization Requirements from Sampling Requirements**: The requirement to plot 20 images at the end does not mean your initial sampling must be limited to 20 images. Because PCA's n_components is strictly constrained by the number of samples ($n\_samples \ge q$), to successfully run $q$ up to 29, we must sample 40 images initially and then filter them using the plotting function. This is a common logical pitfall when coding.

* **Implicit Operations of Black-Box Packages (Automatic Centering and Reconstruction)**: When writing SVD manually, everything must be handled explicitly (manually subtracting and adding back the mean). In contrast, using sklearn's PCA handles centering and reconstruction silently in the background. Without knowing these underlying details, encountering mismatched dimensions or values when using the package can leave you clueless.

<hr>

## Part 4:
There are 70,000 handwritten digit images, each with a size of $28 \times 28$, occupying 54.88 MB of storage space before compression. When performing SVD-based "Rank $q$ approximation," the compression ratio is determined by $q$. Write a program that calculates the compression ratio when adjusting the value of $q$, while simultaneously displaying 100 randomly selected original images and their corresponding reconstructed images for comparison. Additionally, the choice of $q$ can be determined based on the "energy distribution" of $\sigma_1, \sigma_2, \cdots, \sigma_r$; in other words, once $q$ is chosen, the energy ratio (percentage) captured by the selected principal components can be computed. This problem also requires printing out this energy ratio.

In [ ]:
def montage(A, m, n):
    '''
    Create a montage matrix with mn images
    Inputs:
    A: original pxN image matrix with N images (p pixels), N > mn
    m, n: m rows & n columns, total mn images
    Output:
    M: montage matrix containing mn images
    '''
    sz = np.sqrt(A.shape[0]).astype('int') # image size sz x sz
    M = np.zeros((m*sz, n*sz)) # montage image
    for i in range(m) :
        for j in range(n) :
            M[i*sz: (i+1)*sz, j*sz:(j+1)*sz] = \
            A[:, i*n+j].reshape(sz, sz)
    return M

In [ ]:
mnist = loadmat("mnist-original.mat")
X = mnist["data"]
U, E, VT = svd(X, full_matrices = False)
singular = E.cumsum()/E.sum()*100

def show(size,q):
    import numpy as np
    import matplotlib.pyplot as plt
    from scipy.io import loadmat
    #original
    mnist = loadmat("mnist-original.mat")
    X = mnist["data"]
    N, p = X.shape
    idx = np.random.choice(np.arange(70000), size=size, replace=False)
    img = X[:,idx]
    sz = np.sqrt(len(img)).astype('int')
    fig, ax = plt.subplots(1, 2,figsize = (10,15))
    montage_image = montage(img, 10, 10)
    digit_stored = N*p
    ax[0].imshow(montage_image, cmap = 'gray', interpolation = 'nearest')
    ax[0].set_title('Original image')
    ax[0].set_xticks([])
    ax[0].set_yticks([])
    #after
    U, E, VT = svd(montage_image, full_matrices = False)
    Xq = U[:, :q] @ np.diag(E[:q]) @ VT[:q, :]
    print('主成分的能量佔比: {:.2f}'.format(singular[q]/100))
    ax[1].imshow(Xq, cmap = 'gray', interpolation = 'nearest')
    ax[1].set_title('Compression ratio: {:.1f}'.format((N*p)/(q*(N+p))))
    ax[1].set_xticks([])
    ax[1].set_yticks([])
    plt.show()


In [ ]:
show(100,20)

In [ ]:
show(100,40)

In [ ]:
show(100,70)

In [ ]:
show(100,230)

#### Discussion:

When $q$ is relatively small (e.g., $q=20$, capturing an energy ratio of approximately $30\%$), losing too many fine details makes the restored images blurry, and it is hard to see the shapes of the numbers clearly. As $q$ increases (e.g., $q=70$, capturing an energy ratio of $53\%$), the principal components become sufficient to support the primary structures of the digits, allowing the restored images to reach a level of clarity that is easily recognizable by the naked eye. With $q$ increased to $230$, the picture looks almost the same as the original, and the energy ratio reaches $80\%$. Adjusting the value of $q$ directly controls the compression ratio. Although a lower $q$ yields a higher compression rate to save storage space, it comes at the cost of compromised image quality. 

## Part 5:
There are 5 encrypted image files (available for download in a compressed archive). The encryption method is based on the SVD of the Yale Faces matrix $X$ consisting of 2,410 face images of 38 individuals, represented as $X = U\Sigma V^T$. Specifically, $U$ is used as the image encryption tool. Assuming that a vector $x$ represents an original image, the encrypted image is obtained by taking the first $q$ principal components, denoted as $U[:, 0:q]^T x$.

In [ ]:
def show_montage(X, n, m, h, w):
    '''
    X: Image data matrix, where each column represents an image
    n, m: Size of each image (n x m)
    h, w: Create a montage grid with a figure size of (w, h)
    '''
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(h, w, figsize=(w, h))
    if X.shape[1] < w * h: # If the number of images is less than w * h, pad them with zero vectors
        X = np.c_[X, np.zeros((X.shape[0], w*h-X.shape[1]))]
    for i, ax in enumerate(axes.flat):
        ax.imshow(X[:,i].reshape(m, n).T, cmap='gray')
        ax.set_xticks([])
        ax.set_yticks([])
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
import scipy.io
from scipy.io import loadmat
from numpy.linalg import svd
import matplotlib.pyplot as plt

df = pd.read_csv('5 encrypted images.csv') 
df_ = np.array(df)
D = scipy.io.loadmat('allFaces.mat')
X = D['faces'] # each column represents an image
m = int(D['m'][0,0])
n = int(D['n'][0,0])
avgFace = X.mean(axis = 1).reshape(-1,1)
X_avg = X - np.tile(avgFace,(1,X.shape[1]))
U, E, VT = svd(X_avg, full_matrices = False)
q=2000
Xq = U[:,:q] @ df_
show_montage(Xq, n, m ,5, 1)

#### Discussion:

* **Equivalence of Image Dimensionality Reduction and Encryption (Linear Projection)**: 
  * This problem demonstrates the ingenious application of SVD in image processing—projecting high-dimensional images onto the first $q$ principal components (the subspace of the $U$ matrix) is essentially a form of "feature compression and encryption." By setting an appropriate dimension, it is possible to preserve sufficient facial contours and feature details while compressing the data.
  * In practice, we usually do not need to push $q$ to its limit (e.g., all 2410 principal components), because the later principal components often contain only minor noise or very little feature energy. As long as a sufficiently large dimension is set (such as $q=2000$ in this program), we can compress the data while perfectly preserving the vast majority of facial contours and feature details.


* **Reconstruction and Restoration Capability of SVD in Feature Space**: By manually performing data centering (X_avg), calculating the SVD, and using the left singular vectors $U_q$ for back-projection of the compressed encrypted data (i.e., multiplying $U_q$ on the left side of the encrypted image to decrypt it), the process is complete. Finally, through show_montage, 5 encrypted facial images were successfully and clearly restored, and the decrypted images turned out to be the classic old Lenna image. This fully validates the powerful performance of SVD in high-dimensional matrix decomposition and image reconstruction.   